<a href="https://colab.research.google.com/github/22f2000792/MLP-prac/blob/main/hugging_face.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U diffusers transformers accelerate

In [47]:
from transformers import AutoImageProcessor, ResNetForImageClassification
import torch
from PIL import Image
processor = AutoImageProcessor.from_pretrained("microsoft/resnet-50")
model_restNet = ResNetForImageClassification.from_pretrained("microsoft/resnet-50")
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
modelQwen = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", device_map="auto")
def image_classify(image):
    # -----------------------------
    # ResNet
    # -----------------------------
    image = Image.fromarray(image)

    inputs = processor(
        image,
        return_tensors="pt"
    )

    with torch.no_grad():
        logits = model_restNet(**inputs).logits

    predicted_label = logits.argmax(-1).item()

    # Get the actual ResNet label
    label = model_restNet.config.id2label[predicted_label]

    # -----------------------------
    # Qwen
    # -----------------------------
    messages = [
        {
            "role": "user",
            "content": f"Explain more about this: {label}"
        }
    ]

    inputs_qwen = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(modelQwen.device)

    # Move Qwen inputs to Qwen's device

    outputs = modelQwen.generate(
        **inputs_qwen,
        max_new_tokens=100
    )

    # Remove the prompt from the generated output
    generated_tokens = outputs[0][
        inputs_qwen["input_ids"].shape[-1]:
    ]

    out_decoded = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return label, out_decoded



Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [48]:
import gradio as gr
demo = gr.Interface(
    fn=image_classify,
    inputs=gr.Image(type="numpy", label="Upload an Image"),
    outputs=[
        gr.Label(label="ResNet Prediction"),
        gr.Textbox(label="Qwen Explanation",max_lines=8)
    ],
    title="Image Classification with ResNet",
    description="Upload an image and ResNet will classify it."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b80dc73c4f9eeb1d93.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


<think>
Okay, the user is asking me to explain more about the tiger, Panthera tigris. Let me start by recalling what I know. Tigers are large cats, right? They're known for their powerful muscles and impressive speed. But I should mention their physical characteristics first. They have a black-and-turquoise coat, which is a unique feature. Also, they have distinctive spots, which are part of their camouflage.

I should explain their habitats. Tigers live in various regions, including Asia, Africa, and the Americas. Different countries have different populations. For example, in India, they're common in the Indian subcontinent, while in Africa, they're found in the savannahs. Maybe mention the conservation status too, since it's important to note their role in the ecosystem.

Wait, the user wants more details. Should I include their diet? Tigers are carnivores, feeding on prey like prey. Also, their social behavior. Are they solitary or do they live in groups? I think they're generally 